In [61]:
import sys
sys.path.append("../")

In [62]:
import pandas as pd
import plotly.graph_objects as go
from technicals.indicators import RSI
from technicals.patterns import apply_patterns
from plotting import CandlePlot
import datetime as dt


In [63]:
df_raw = pd.read_pickle("../data/EUR_USD_H1.pkl")

In [64]:
df_raw.shape

(54516, 14)

In [65]:
df_an = df_raw.copy()#df_raw.iloc[-6000:].copy()
df_an.reset_index(drop=True, inplace=True)

In [66]:
df_an.shape

(54516, 14)

In [67]:
df_an = RSI(df_an)

In [68]:
df_an.tail()

,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c,RSI_14
54511,2024-10-09 19:00:00+00:00,3634,1.09366,1.09415,1.09366,1.09387,1.09359,1.09408,1.09358,1.09379,1.09374,1.09423,1.09374,1.09395,31.865151
54512,2024-10-09 20:00:00+00:00,1765,1.09388,1.09410,1.09368,1.09400,1.09379,1.09402,1.09359,1.09393,1.09396,1.09417,1.09376,1.09408,33.413564
54513,2024-10-09 21:00:00+00:00,1339,1.09386,1.09430,1.09382,1.09416,1.09374,1.09415,1.09364,1.09403,1.09399,1.09445,1.09399,1.09430,35.360609
54514,2024-10-09 22:00:00+00:00,966,1.09420,1.09426,1.09380,1.09408,1.09411,1.09414,1.09371,1.09401,1.09430,1.09440,1.09389,1.09416,34.812484
54515,2024-10-09 23:00:00+00:00,1406,1.09409,1.09422,1.09390,1.09416,1.09401,1.09415,1.09382,1.09408,1.09417,1.09431,1.09398,1.09425,35.882818


In [69]:
df_an = apply_patterns(df_an)

In [70]:
df_an['EMA_200'] = df_an.mid_c.ewm(span=200, min_periods=200).mean()

In [71]:
df_an.columns

Index(['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h',
       'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c', 'RSI_14',
       'body_lower', 'body_upper', 'body_bottom_perc', 'body_top_perc',
       'body_perc', 'direction', 'body_size', 'low_change', 'high_change',
       'body_size_change', 'mid_point', 'mid_point_prev_2', 'body_size_prev',
       'direction_prev', 'direction_prev_2', 'body_perc_prev',
       'body_perc_prev_2', 'HANGING_MAN', 'SHOOTING_STAR', 'SPINNING_TOP',
       'MARUBOZU', 'ENGULFING', 'TWEEZER_TOP', 'TWEEZER_BOTTOM',
       'MORNING_STAR', 'EVENING_STAR', 'EMA_200'],
      dtype='object')

In [72]:
our_cols = ['time', 'mid_o', 'mid_h', 'mid_l', 'mid_c',
            'bid_o', 'bid_h','bid_l', 'bid_c', 
            'ask_o', 'ask_h', 'ask_l', 'ask_c',
                 'ENGULFING', 'direction', 'EMA_200', 'RSI_14' ]

In [73]:
df_slim = df_an[our_cols].copy()
df_slim.dropna(inplace=True)
df_slim.reset_index(drop=True, inplace=True)

In [74]:
df_slim.head()

,time,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c,ENGULFING,direction,EMA_200,RSI_14
0,2016-01-19 07:00:00+00:00,1.08736,1.08764,1.08595,1.08652,1.08726,1.08756,1.08587,1.08645,1.08745,1.08772,1.08603,1.08660,False,-1,1.088334,36.239350
1,2016-01-19 08:00:00+00:00,1.08654,1.08846,1.08636,1.08846,1.08647,1.08832,1.08629,1.08832,1.08662,1.08861,1.08644,1.08861,True,1,1.088336,47.615598
2,2016-01-19 09:00:00+00:00,1.08844,1.08890,1.08702,1.08724,1.08834,1.08883,1.08694,1.08699,1.08854,1.08898,1.08710,1.08749,False,-1,1.088323,42.482293
3,2016-01-19 10:00:00+00:00,1.08728,1.08834,1.08662,1.08730,1.08702,1.08827,1.08655,1.08722,1.08755,1.08842,1.08668,1.08737,False,1,1.088312,42.808846
4,2016-01-19 11:00:00+00:00,1.08728,1.08732,1.08599,1.08630,1.08721,1.08726,1.08593,1.08623,1.08734,1.08738,1.08605,1.08637,True,-1,1.088289,38.849941


In [75]:
BUY = 1
SELL = -1
NONE = 0
RSI_LIMIT = 50.0

def apply_signal(row):
    if row.ENGULFING == True:
        if row.direction == BUY and row.mid_l > row.EMA_200:
            if row.RSI_14 > RSI_LIMIT:
                return BUY
        if row.direction == SELL and row.mid_h < row.EMA_200:
            if row.RSI_14 < RSI_LIMIT:
                return SELL
    return NONE      

In [76]:
df_slim["SIGNAL"] = df_slim.apply(apply_signal, axis=1)

In [77]:
df_slim["SIGNAL"].value_counts()

SIGNAL
 0    49511
-1     2406
 1     2400
Name: count, dtype: int64

In [78]:
LOSS_FACTOR = -1.0
PROFIT_FACTOR = 1.5

def apply_take_profit(row):
    if row.SIGNAL != NONE:
        if row.SIGNAL == BUY:
            return (row.ask_c - row.ask_o) * PROFIT_FACTOR + row.ask_c
        else:
            return (row.bid_c - row.bid_o) * PROFIT_FACTOR + row.bid_c
    else:
        return 0.0

def apply_stop_loss(row):
    if row.SIGNAL != NONE:
        if row.SIGNAL == BUY:
            return row.ask_o
            
        else:    
            return row.bid_o
    else:
        return 0.0

In [79]:
df_slim["TP"] = df_slim.apply(apply_take_profit, axis=1)
df_slim["SL"] = df_slim.apply(apply_stop_loss, axis=1)

In [80]:
df_slim[df_slim.SIGNAL==BUY].head()

,time,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c,ENGULFING,direction,EMA_200,RSI_14,SIGNAL,TP,SL
15,2016-01-19 22:00:00+00:00,1.09060,1.09143,1.09052,1.09140,1.09010,1.09130,1.09010,1.09127,1.09110,1.09156,1.09063,1.09153,True,1,1.088541,57.545307,1,1.092175,1.09110
17,2016-01-20 00:00:00+00:00,1.09112,1.09246,1.09100,1.09214,1.09104,1.09239,1.09091,1.09206,1.09120,1.09255,1.09108,1.09223,True,1,1.088610,59.970073,1,1.093775,1.09120
23,2016-01-20 06:00:00+00:00,1.09491,1.09635,1.09491,1.09546,1.09484,1.09628,1.09484,1.09538,1.09498,1.09643,1.09498,1.09554,True,1,1.088985,68.866726,1,1.096380,1.09498
25,2016-01-20 08:00:00+00:00,1.09398,1.09760,1.09397,1.09625,1.09390,1.09754,1.09390,1.09619,1.09405,1.09768,1.09403,1.09631,True,1,1.089121,67.087323,1,1.099700,1.09405
146,2016-01-27 09:00:00+00:00,1.08673,1.08764,1.08665,1.08726,1.08666,1.08756,1.08658,1.08719,1.08680,1.08771,1.08672,1.08732,True,1,1.085983,60.955020,1,1.088100,1.08680


In [81]:
df_plot = df_slim.iloc[0:40]
cp = CandlePlot(df_plot, candles=True)

trades = cp.df_plot[cp.df_plot.SIGNAL != NONE]

markers = ['mid_c', 'TP', 'SL']
marker_colors = ['#0000FF', '#00FF00', '#FF0000']

for i in range(3):
    cp.fig.add_trace(go.Scatter(
        x = trades.sTime,
        y = trades[markers[i]],
        mode = 'markers',
        marker=dict(color=marker_colors[i], size=12)
    ))

cp.show_plot(line_traces=["EMA_200"], sec_traces=['RSI_14'], height=250)

In [82]:
class Trade:
    def __init__(self, row):
        self.running = True
        self.start_index = row.name
        
        if row.SIGNAL == BUY:
            self.start_price = row.ask_c
            self.trigger_price = row.ask_c
        else:
            self.start_price = row.bid_c
            self.trigger_price = row.bid_c
            
        self.SIGNAL = row.SIGNAL
        self.TP = row.TP
        self.SL = row.SL
        self.result = 0.0
        self.end_time = row.time
        self.start_time = row.time
        self.duration = 0
        
    def close_trade(self, row, result, trigger_price):
        self.running = False
        self.result = result
        self.end_time = row.time
        self.trigger_price = trigger_price
        
    def update(self, row):
        self.duration += 1
        if self.SIGNAL == BUY:
            if row.bid_h >= self.TP:
                self.close_trade(row, PROFIT_FACTOR, row.bid_h)
            elif row.bid_l <= self.SL:
                self.close_trade(row, LOSS_FACTOR, row.bid_l)
        if self.SIGNAL == SELL:
            if row.ask_l <= self.TP:
                self.close_trade(row, PROFIT_FACTOR, row.ask_l)
            elif row.ask_h >= self.SL:
                self.close_trade(row, LOSS_FACTOR, row.ask_h)   

In [83]:
open_trades = []
closed_trades = []

for index, row in df_slim.iterrows():
    for ot in open_trades:
        ot.update(row)
        if ot.running == False:
            closed_trades.append(ot)
    open_trades = [x for x in open_trades if x.running == True]
    
    if row.SIGNAL != NONE:
        open_trades.append(Trade(row))    

In [84]:
df_results = pd.DataFrame.from_dict([vars(x) for x in closed_trades])

In [85]:
df_results.result.sum()

np.float64(-694.0)

In [86]:
df_results.sort_values(by="start_index", inplace=True)

In [87]:
df_m5 = pd.read_pickle("../data/EUR_USD_M5.pkl")

In [88]:
df_m5.shape

(652472, 14)

In [ ]:
df_m5.time.max()

In [ ]:
df_raw.time.max()

In [ ]:
from dateutil import parser

In [ ]:
time_min = parser.parse("2021-12-15T10:00:00Z")
time_max = parser.parse("2021-12-15T11:00:00Z")
df_m5_s = df_m5[(df_m5.time>=time_min)&(df_m5.time<=time_max)]
df_raw_s = df_raw[(df_raw.time>=time_min)&(df_raw.time<=time_max)]

In [ ]:
df_m5_s

In [ ]:
df_raw_s

In [89]:
df_m5_slim = df_m5[['time','bid_h', 'bid_l','ask_h', 'ask_l' ]].copy()

In [90]:
df_m5_slim.head()

,time,bid_h,bid_l,ask_h,ask_l
0,2016-01-07 00:00:00+00:00,1.07802,1.07750,1.07820,1.07768
1,2016-01-07 00:05:00+00:00,1.07811,1.07755,1.07827,1.07772
2,2016-01-07 00:10:00+00:00,1.07823,1.07803,1.07840,1.07822
3,2016-01-07 00:15:00+00:00,1.07821,1.07790,1.07838,1.07807
4,2016-01-07 00:20:00+00:00,1.07789,1.07768,1.07809,1.07784


In [91]:
df_signals = df_slim[df_slim.SIGNAL != NONE].copy()

In [92]:
df_signals['m5_start'] = [x + dt.timedelta(hours=1) for x in df_signals.time]

In [93]:
df_signals['start_index_h1'] = df_signals.index 

In [94]:
df_signals.head()

,time,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,...,ask_c,ENGULFING,direction,EMA_200,RSI_14,SIGNAL,TP,SL,m5_start,start_index_h1
4,2016-01-19 11:00:00+00:00,1.08728,1.08732,1.08599,1.08630,1.08721,1.08726,1.08593,1.08623,1.08734,...,1.08637,True,-1,1.088289,38.849941,-1,1.084760,1.08721,2016-01-19 12:00:00+00:00,4
15,2016-01-19 22:00:00+00:00,1.09060,1.09143,1.09052,1.09140,1.09010,1.09130,1.09010,1.09127,1.09110,...,1.09153,True,1,1.088541,57.545307,1,1.092175,1.09110,2016-01-19 23:00:00+00:00,15
17,2016-01-20 00:00:00+00:00,1.09112,1.09246,1.09100,1.09214,1.09104,1.09239,1.09091,1.09206,1.09120,...,1.09223,True,1,1.088610,59.970073,1,1.093775,1.09120,2016-01-20 01:00:00+00:00,17
23,2016-01-20 06:00:00+00:00,1.09491,1.09635,1.09491,1.09546,1.09484,1.09628,1.09484,1.09538,1.09498,...,1.09554,True,1,1.088985,68.866726,1,1.096380,1.09498,2016-01-20 07:00:00+00:00,23
25,2016-01-20 08:00:00+00:00,1.09398,1.09760,1.09397,1.09625,1.09390,1.09754,1.09390,1.09619,1.09405,...,1.09631,True,1,1.089121,67.087323,1,1.099700,1.09405,2016-01-20 09:00:00+00:00,25


In [95]:
df_signals.columns

Index(['time', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l',
       'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c', 'ENGULFING', 'direction',
       'EMA_200', 'RSI_14', 'SIGNAL', 'TP', 'SL', 'm5_start',
       'start_index_h1'],
      dtype='object')

In [96]:
df_signals.drop(['time', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l',
        'ask_o', 'ask_h', 'ask_l', 'ENGULFING', 'direction',
       'EMA_200', 'RSI_14'], axis=1, inplace=True)

In [97]:
df_signals.head()

,bid_c,ask_c,SIGNAL,TP,SL,m5_start,start_index_h1
4,1.08623,1.08637,-1,1.084760,1.08721,2016-01-19 12:00:00+00:00,4
15,1.09127,1.09153,1,1.092175,1.09110,2016-01-19 23:00:00+00:00,15
17,1.09206,1.09223,1,1.093775,1.09120,2016-01-20 01:00:00+00:00,17
23,1.09538,1.09554,1,1.096380,1.09498,2016-01-20 07:00:00+00:00,23
25,1.09619,1.09631,1,1.099700,1.09405,2016-01-20 09:00:00+00:00,25


In [98]:
df_signals.rename(columns={
    'bid_c' : 'start_price_BUY',
    'ask_c' : 'start_price_SELL',
    'm5_start' : 'time'
}, inplace=True)

In [99]:
df_signals.head(2)

,start_price_BUY,start_price_SELL,SIGNAL,TP,SL,time,start_index_h1
4,1.08623,1.08637,-1,1.084760,1.08721,2016-01-19 12:00:00+00:00,4
15,1.09127,1.09153,1,1.092175,1.09110,2016-01-19 23:00:00+00:00,15


In [100]:
df_m5_slim.head(2)

,time,bid_h,bid_l,ask_h,ask_l
0,2016-01-07 00:00:00+00:00,1.07802,1.07750,1.07820,1.07768
1,2016-01-07 00:05:00+00:00,1.07811,1.07755,1.07827,1.07772


In [101]:
merged = pd.merge(left=df_m5_slim, right=df_signals, on='time', how='left')

In [102]:
merged.fillna(0, inplace=True)

In [103]:
merged.SIGNAL = merged.SIGNAL.astype(int)
merged.start_index_h1 = merged.start_index_h1.astype(int)

In [104]:
merged.head()

,time,bid_h,bid_l,ask_h,ask_l,start_price_BUY,start_price_SELL,SIGNAL,TP,SL,start_index_h1
0,2016-01-07 00:00:00+00:00,1.07802,1.07750,1.07820,1.07768,0.0,0.0,0,0.0,0.0,0
1,2016-01-07 00:05:00+00:00,1.07811,1.07755,1.07827,1.07772,0.0,0.0,0,0.0,0.0,0
2,2016-01-07 00:10:00+00:00,1.07823,1.07803,1.07840,1.07822,0.0,0.0,0,0.0,0.0,0
3,2016-01-07 00:15:00+00:00,1.07821,1.07790,1.07838,1.07807,0.0,0.0,0,0.0,0.0,0
4,2016-01-07 00:20:00+00:00,1.07789,1.07768,1.07809,1.07784,0.0,0.0,0,0.0,0.0,0


In [105]:
class TradeM5:
    def __init__(self, row):
        self.running = True
        self.start_index_m5 = row.name
        self.start_index_h1 = row.start_index_h1
        
        if row.SIGNAL == BUY:
            self.start_price = row.start_price_BUY
            self.trigger_price = row.start_price_BUY
        else:
            self.start_price = row.start_price_SELL
            self.trigger_price = row.start_price_SELL
            
        self.SIGNAL = row.SIGNAL
        self.TP = row.TP
        self.SL = row.SL
        self.result = 0.0
        self.end_time = row.time
        self.start_time = row.time
        self.duration = 1
        
    def close_trade(self, row, result, trigger_price):
        self.running = False
        self.result = result
        self.end_time = row.time
        self.trigger_price = trigger_price
        
    def update(self, row):
        self.duration += 1
        if self.SIGNAL == BUY:
            if row.bid_h >= self.TP:
                self.close_trade(row, PROFIT_FACTOR, row.bid_h)
            elif row.bid_l <= self.SL:
                self.close_trade(row, LOSS_FACTOR, row.bid_l)
        if self.SIGNAL == SELL:
            if row.ask_l <= self.TP:
                self.close_trade(row, PROFIT_FACTOR, row.ask_l)
            elif row.ask_h >= self.SL:
                self.close_trade(row, LOSS_FACTOR, row.ask_h)   

In [ ]:
print(merged.columns)

In [106]:
open_trades_m5 = []
closed_trades_m5 = []

for index, row in merged.iterrows():
    
    if row.SIGNAL != NONE:
        open_trades_m5.append(TradeM5(row))   
        
    for ot in open_trades_m5:
        ot.update(row)
        if ot.running == False:
            closed_trades_m5.append(ot)
    open_trades_m5 = [x for x in open_trades_m5 if x.running == True]


In [107]:
df_res_m5 = pd.DataFrame.from_dict([vars(x) for x in closed_trades_m5])

In [108]:
df_res_m5.head()

,running,start_index_m5,start_index_h1,start_price,trigger_price,SIGNAL,TP,SL,result,end_time,start_time,duration
0,False,2445,4,1.08637,1.08746,-1,1.084760,1.08721,-1.0,2016-01-19 12:05:00+00:00,2016-01-19 12:00:00+00:00,3
1,False,2577,15,1.09127,1.09102,1,1.092175,1.09110,-1.0,2016-01-19 23:00:00+00:00,2016-01-19 23:00:00+00:00,2
2,False,2601,17,1.09206,1.09404,1,1.093775,1.09120,1.5,2016-01-20 03:00:00+00:00,2016-01-20 01:00:00+00:00,26
3,False,2673,23,1.09538,1.09488,1,1.096380,1.09498,-1.0,2016-01-20 07:20:00+00:00,2016-01-20 07:00:00+00:00,6
4,False,2697,25,1.09619,1.09362,1,1.099700,1.09405,-1.0,2016-01-20 09:20:00+00:00,2016-01-20 09:00:00+00:00,6


In [109]:
df_res_m5.result.sum()

np.float64(-1419.5)

: 